In [ ]:
# Step 1: Import the building blocks
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages


# Step 2: Define the state schema (the "blank form")
class AgentState(TypedDict):
 messages: Annotated[list, add_messages]             # Conversation history
 user_name: str                                     # Who is the user?
 current_task: str                                   # What is the agent doing?
 task_status: str

In [ ]:
from langchain_core.messages import AIMessage

# The "worker" — reads the form, does work, writes back changes
def chat_node(state: AgentState):
  latest_text = state["messages"][-1].content               # READ the form
  reply = f"I heard: {latest_text}. How can I help?"

  return {                                      # WRITE back only what changed
  "messages": [AIMessage(content=reply)],
  "task_status": "in_progress"
  }

In [ ]:
from langgraph.graph import StateGraph, START, END


graph = StateGraph(AgentState)

graph.add_node("chat" , chat_node)

graph.add_edge(START, "chat")
graph.add_edge("chat", END)

app = graph.compile()

In [ ]:
from langchain_core.messages import HumanMessage

# This is the ONLY line you write to start a conversation:
#result = app.invoke("Hi")
result = app.invoke({"messages": [HumanMessage(content="Hi")]})
result

{'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='5458b2ab-a952-4783-9d9f-0d21e94bddab'),
  AIMessage(content='I heard: Hi. How can I help?', additional_kwargs={}, response_metadata={}, id='2befc6d4-dd69-4086-90bd-45b6a4f6d1d3', tool_calls=[], invalid_tool_calls=[])],
 'task_status': 'in_progress'}

In [ ]:
print(result['messages'][0].content)
print(result['messages'][1].content)

Hi
I heard: Hi. How can I help?


In [ ]:
print(result['task_status'])
print(result['task_status'])

in_progress
in_progress


In [ ]:
# ─── STEP 1: IMPORTS ────────────────────────────────────────
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
# ─── STEP 2: STATE SCHEMA (The Booking Form) ───────────────
class BookingState(TypedDict):
 messages: Annotated[list, add_messages]          # Chat history → APPEND
 passenger_name: str                          # Who is flying? → OVERWRITE
 departure_city: str                            # From where? → OVERWRITE
 arrival_city: str                              # To where? → OVERWRITE
 travel_date: str                                     # When? → OVERWRITE
 booking_status: str                            # Current progress → OVERWRITE

In [ ]:
# ─── STEP 3: NODE FUNCTIONS (The Workers) ───────────────────
def collect_info_node(state: BookingState):
  """Greet the user and collect booking details from their message."""
  user_msg = state["messages"][-1].content if state["messages"] else ""

  # Simple keyword extraction (in real agent, an LLM would do this)
  name = "Priya" if "priya" in user_msg.lower() else state.get("passenger_name", "")
  departure = "Delhi" if "delhi" in user_msg.lower() else state.get("departure_city", "")
  arrival = "Mumbai" if "mumbai" in user_msg.lower() else state.get("arrival_city", "")
  date = "15 May" if "may" in user_msg.lower() else state.get("travel_date", "")
  reply = (f"Got it! Booking for {name}, "
  f"{departure} → {arrival} on {date}. Confirming...")

  return {
    "messages": [AIMessage(content=reply)],         # Appended by reducer
    "passenger_name": name,                         # Overwritten
    "departure_city": departure,                  # Overwritten
    "arrival_city": arrival,                      # Overwritten
    "travel_date": date,                            # Overwritten
    "booking_status": "details_collected",            # Overwritten
  }

In [ ]:
def confirm_booking_node(state: BookingState):

    """Confirm the booking and generate a PNR."""
    name = state.get("passenger_name", "Guest")
    dep = state.get("departure_city", "?")
    arr = state.get("arrival_city", "?")
    date = state.get("travel_date", "?")
    confirmation = (f"Booking CONFIRMED!\n"

    f"Passenger: {name}\n"
    f"Route: {dep} → {arr}\n"
    f"Date: {date}\n"
    f"PNR: SKY-98452")

    return {
    "messages": [AIMessage(content=confirmation)], # Appended
    "booking_status": "confirmed", # Overwritten
 }

In [ ]:
graph = StateGraph(BookingState)

graph.add_node("collect_info", collect_info_node)
graph.add_node("confirm", confirm_booking_node)

graph.add_edge(START, "collect_info")               # Start → Collect
graph.add_edge("collect_info", "confirm")             # Collect → Confirm
graph.add_edge("confirm", END)                        # Confirm → End

app = graph.compile()

In [ ]:
# ─── STEP 5: INVOKE ─────────────────────────────────────────
result = app.invoke({
  "messages": [HumanMessage(
  #content="I want to travel from delhi to mumbai , my name is Bipul"
  content="Hi, I'm Bipul. Book Jaipur to Mumbai on 25 May."
  )]
})

In [ ]:
for msg in result["messages"]:
 role = "USER" if isinstance(msg, HumanMessage) else "BOT"
 print(f"[{role}] {msg.content}")

print(f"\nBooking Status: {result['booking_status']}")
print(f"Passenger: {result['passenger_name']}")
print(f"Route: {result['departure_city']} → {result['arrival_city']}")
print(f"Date: {result['travel_date']}")

[USER] Hi, I'm Bipul. Book Jaipur to Mumbai on 25 May.
[BOT] Got it! Booking for ,  → Mumbai on 15 May. Confirming...
[BOT] Booking CONFIRMED!
Passenger: 
Route:  → Mumbai
Date: 15 May
PNR: SKY-98452

Booking Status: confirmed
Passenger: 
Route:  → Mumbai
Date: 15 May


In [ ]:
import sys
from platform import python_version
print(python_version())

3.12.13
